In [ ]:
#mounting google drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Step 1: Bipartite Graph

Construction bipartite asset-owner graph: edges only with different types, with all assets on one side and all wallets on the other.

In [ ]:
import pandas as pd
df = pd.read_json("fragment_final.jsonl", lines=True)
df

,Asset,Scrape date,Ownership history,Bids,Resale,Highest bid,grado_asset
0,@dubaihouse,2025-09-30 02:34:42,"[{'Owner': 'conig.t.me', 'Date of Sale': '2024...",[],None,NaN,1.0
1,@totln,2025-11-16 20:42:28,[{'Owner': 'UQBB5t1vKAJMgLj-5cGHXhxqweHSxNkm0z...,[],None,NaN,1.0
2,@hangzhoubot,2025-11-13 06:50:16,"[{'Owner': 'mahua.t.me', 'Date of Sale': '2025...",[],None,NaN,4.0
3,@gssssss,2025-11-13 06:34:48,[{'Owner': 'UQAVWbDUur7WDLwT71dzKf5RDZfTUHTnWE...,[],None,NaN,2.0
4,@kefu586,2025-11-13 09:12:22,[{'Owner': 'UQBC6_I2vvLCUfwNnfSGlrVTmWG7PO1j37...,[],None,NaN,1.0
...,...,...,...,...,...,...,...
110678,@milklover,2025-11-14 13:10:00,"[{'Owner': 'sara88.t.me', 'Date of Sale': '202...",[],None,NaN,8.0
110679,@kachuguard,2025-11-13 09:00:12,[{'Owner': 'UQA_KHeQZjTPfC-4e5wrVu1WOcV4Lv5D4a...,[{'Bidder': 'UQA_KHeQZjTPfC-4e5wrVu1WOcV4Lv5D4...,None,NaN,1.0
110680,@daili888,2025-11-13 03:01:33,"[{'Owner': 'tantan.t.me', 'Date of Sale': '202...",[],None,NaN,1.0
110681,@hackn,2025-05-05 16:52:12,[],[{'Bidder': 'UQBoifQu7PJFOaVjlkRB3l8b2M6LyItRW...,0.0,104.0,1.0


In [ ]:
import networkx as nx

#create empty graph
G = nx.Graph()

#iterate through the dataframe
for index, row in df.iterrows():
    #take the asset and add it to the graph if it doesn't exist already
    asset = row["Asset"]
    if asset not in G:
        G.add_node(asset, bipartite=0, node_type="asset")

    #take all the wallets from ownership history and bids
    wallets = set() #start with an empty set
    #wallets from ownership history
    for entry in row.get("Ownership history", []): #if column doesn't exist, get an empty list. better than row["Ownership history"].
        owner = entry.get("Owner") #get owner key value from entry (which is a dictionary)
        if owner:
            wallets.add(owner)
    #wallets from bids
    for entry in row.get("Bids", []):
        bidder = entry.get("Bidder")
        if bidder:
            wallets.add(bidder)

    #for each wallet, add it to the graph if it doesn't exist already and create a connection with the current asset
    for wallet in wallets:
        if wallet not in G:
            G.add_node(wallet, bipartite=1, node_type='wallet')

        #add arc
        G.add_edge(asset, wallet)

In [ ]:
#calculate number of nodes
#total
print(f"total number of nodes: {G.number_of_nodes():,}")
#assets
asset_nodes = {n for n, d in G.nodes(data=True) if d.get('bipartite') == 0}
print(f"number of asset nodes:  {len(asset_nodes):,}")
#wallets
wallet_nodes = {n for n, d in G.nodes(data=True) if d.get('bipartite') == 1}
print(f"number of wallet nodes: {len(wallet_nodes):,}")

#connections
print(f"total connections: {G.number_of_edges():,}")

total number of nodes: 161,491
number of asset nodes:  110,683
number of wallet nodes: 50,808
total connections: 281,654


#Step 2: GCC extraction

Extraction of the giant connected component, which will be used from now on as the network.

In [ ]:
#find all the connected components
components = list(nx.connected_components(G))
#number of connected components
print(f"connected components: {len(components)}")

connected components: 5091


In [ ]:
#find gcc
gcc_nodes = max(components, key=len)
#gcc size
print(f"gcc number of nodes: {len(gcc_nodes):} - {len(gcc_nodes)/G.number_of_nodes()*100:.1f}% of the previous total ({G.number_of_nodes():,})")

gcc number of nodes: 150421 - 93.1% of the previous total (161,491)


In [ ]:
#create a new graph with only gcc nodes
G_gcc = G.subgraph(gcc_nodes).copy()
#print number of nodes and connections
print(f"gcc nodes: {G_gcc.number_of_nodes():,}")
print(f"gcc connections: {G_gcc.number_of_edges():,} - {G_gcc.number_of_edges()/G.number_of_edges()*100:.1f}% of the previous total ({G.number_of_edges():,})")

gcc nodes: 150,421
gcc connections: 275,330 - 97.8% of the previous total (281,654)


In [ ]:
#find asset and wallet nodes in gcc
asset_nodes_gcc = {n for n, d in G_gcc.nodes(data=True) if d.get('bipartite') == 0}
wallet_nodes_gcc = {n for n, d in G_gcc.nodes(data=True) if d.get('bipartite') == 1}
print(f"number of assets:  {len(asset_nodes_gcc):,}")
print(f"number of wallets: {len(wallet_nodes_gcc):,}")

number of assets:  102,649
number of wallets: 47,772


In [ ]:
#check other components
sorted_components = sorted(components, key=len, reverse=True)
print(f"top 5 components (sorted in decreasing size):")
for i, comp in enumerate(sorted_components[:5]):
    print(f"  {i+1}. {len(comp):,} nodes")

top 5 components (sorted in decreasing size):
  1. 150,421 nodes
  2. 276 nodes
  3. 50 nodes
  4. 47 nodes
  5. 44 nodes


In [ ]:
#create filtered dataframe with only assets in gcc
df_gcc = df[df['Asset'].isin(asset_nodes_gcc)].copy()

print(f"original df length: {len(df):,}")
print(f"gcc dataframe length: {len(df_gcc):,}")

original df length: 110,683
gcc dataframe length: 102,649


## Step 3: Asset-Asset Projection

Asset-asset projection: create an asset-only graph, where there is an edge between assets that share a wallet, with the number of shared wallets being the weight of the connection.

In [ ]:
from networkx.algorithms import bipartite

print(f"bipartite GCC graph: {G_gcc.number_of_nodes():,} nodes, {G_gcc.number_of_edges():,} edges")
print("\n")

#weighted projection (weight = number of common assets)
G_proj = bipartite.weighted_projected_graph(G_gcc, asset_nodes_gcc)

print("asset-asset projection:")
print(f"nodes (assets): {G_proj.number_of_nodes():,}")
print(f"connections (edges): {G_proj.number_of_edges():,}")

density = nx.density(G_proj)
weights = [d['weight'] for u, v, d in G_proj.edges(data=True)]

print(f"network density: {density:.6f}")
print(f"weight distribution (shared wallets):")
print(f"Min: {min(weights)} | Max: {max(weights)} | Mean: {sum(weights)/len(weights):.2f}")

#gcc extraction from projection
components_proj = list(nx.connected_components(G_proj))
print(f"connected components found: {len(components_proj)}")

bipartite GCC graph: 150,421 nodes, 275,330 edges


asset-asset projection:
nodes (assets): 102,649
connections (edges): 30,609,071
network density: 0.005810
weight distribution (shared wallets):
Min: 1 | Max: 19 | Mean: 1.11
connected components found: 1


In [ ]:
import gc

if len(components_proj) > 0:
    gcc_proj_nodes = max(components_proj, key=len)

    #identify nodes not in the gcc (so to remove)
    nodes_to_remove = set(G_proj.nodes()) - set(gcc_proj_nodes)
    #cancel from G_proj the nodes to remove
    G_proj.remove_nodes_from(nodes_to_remove)
    #G_proj is now the GCC
    G_proj_final = G_proj

    #build a set to filter the df
    assets_in_proj = set(G_proj_final.nodes())

    del nodes_to_remove
    del components_proj
    gc.collect()

    print(f"number of assets in the gcc: {len(assets_in_proj):,}")
else:
    G_proj_final = G_proj
    assets_in_proj = set(G_proj_final.nodes())

#update the df to remove non-gcc nodes
df_final = df_gcc[df_gcc['Asset'].isin(assets_in_proj)].copy()
gc.collect()

print(f"total assets in df_final: {len(df_final):,}")

number of assets in the gcc: 102,649
total assets in df_final: 102,649


#Step 4: Linguistic Features Extraction

Extraction of basic linguistic features needed to construct the clusters in step 5.

##Step 4.1: Basic Linguistic Features

In [ ]:
import pandas as pd
import re
import math
from collections import Counter

#function to calculate shannon entropy
def calculate_shannon_entropy(text):
    if not text:
        return 0
    #count the frequency of each character
    probs = [n / len(text) for n in Counter(text).values()]
    #shannon formula: H = -sum(p*log2(p))
    return -sum(p * math.log2(p) for p in probs)

def extract_linguistic_features(username):
    if not isinstance(username, str):
        return None

    text = username[1:] if username.startswith('@') else username

    #basic counts
    length = len(text)
    n_alpha = sum(c.isalpha() for c in text)
    n_digit = sum(c.isdigit() for c in text)
    n_special = length - n_alpha - n_digit

    #percentages count
    pct_alpha = n_alpha / length if length > 0 else 0
    pct_digit = n_digit / length if length > 0 else 0

    #specific characters
    has_underscore = '_' in text
    n_underscore = text.count('_')

    #case
    is_lowercase = text.islower()

    #digit positioning
    starts_with_digit = text[0].isdigit() if text else False
    ends_with_digit = text[-1].isdigit() if text else False

    #repetition present
    has_repetition = bool(re.search(r'(.)\1{2,}', text.lower()))

    #check for crypto keywords by initializing a custom set
    crypto_keywords = {'btc', 'bitcoin', 'eth', 'ethereum', 'sol', 'solana', 'ton', 'toncoin', 'bnb', 'binance', 'ada', 'cardano', 'dot',
    'polkadot', 'xrp', 'doge', 'dogecoin', 'shib', 'shiba', 'coin', 'token', 'wallet', 'dao', 'nft', 'defi', 'dex', 'cex', 'chain', 'blockchain',
    'web3', 'dapp', 'node', 'validator', 'bridge', 'gas', 'ledger', 'swap', 'yield', 'farm', 'stake', 'staking', 'pool', 'liquidity', 'arbitrage',
    'long', 'short', 'moon', 'lambo', 'pump', 'dump', 'hodl', 'fomo', 'alpha', 'mint', 'airdrop', 'whitelist', 'wl', 'ido', 'ico', 'supply',
    'burn', 'halving', 'stable', 'usdt', 'usdc', 'ens', 'domain', 'handle', 'address', 'cryp'}
    has_crypto = any(kw in text.lower() for kw in crypto_keywords)

    #shannon entropy
    entropy = calculate_shannon_entropy(text)

    #ttr
    ttr = len(set(text)) / len(text) if len(text) > 0 else 0

    return {
        'length': length,
        'n_alpha': n_alpha,
        'n_digit': n_digit,
        'n_special': n_special,
        'pct_alpha': pct_alpha,
        'pct_digit': pct_digit,
        'has_underscore': has_underscore,
        'is_lowercase': is_lowercase,
        'starts_with_digit': starts_with_digit,
        'ends_with_digit': ends_with_digit,
        'has_repetition': has_repetition,
        'has_crypto': has_crypto,
        'entropy': entropy,
        'ttr': ttr
    }

df_features = df['Asset'].apply(extract_linguistic_features).apply(pd.Series)
df_final = df_final.join(df_features)


##Step 4.2: Embeddings

Embedding extraction with BGE-M3 (1024 dimensions) model.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('BAAI/bge-m3')
#prepare usernames
usernames_to_embed = df_final['Asset'].str.replace('@', '', regex=False).tolist()
#generate embeddings
embeddings = model.encode(usernames_to_embed, show_progress_bar=True, batch_size=32)
#construct embeddings dataframe and join
embedding_cols = [f'emb_{i}' for i in range(embeddings.shape[1])]
df_embeddings = pd.DataFrame(embeddings, columns=embedding_cols, index=df_final.index)
df_final = df_final.join(df_embeddings)

df_final.to_csv('df_bgem3.csv', index=False)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/3208 [00:00<?, ?it/s]

##Step 4.3: Standardization

Embeddings have values between -1 and 1, so we conduct a standardization on non-embedding linguistic features.

In [ ]:
from sklearn.preprocessing import StandardScaler

linguistic_features = [
    'length', 'n_alpha', 'n_digit', 'n_special', 'pct_alpha', 'pct_digit',
    'has_underscore', 'is_lowercase', 'starts_with_digit', 'ends_with_digit',
    'has_repetition', 'has_crypto', 'entropy', 'ttr'
]

scaler = StandardScaler()
scaled_manual = scaler.fit_transform(df_final[linguistic_features])
#data from embeddings
embedding_data = df_final[[f'emb_{i}' for i in range(384)]].values
#X is the final matrix with all the features
X = np.hstack((scaled_manual, embedding_data))

X.shape

(102649, 398)

##Step 4.4: Dimensionality reduction

Dimensionality reduction is conducted in order to reduce the number of dimensions and facilitate the clustering (otherwise, the embedding features would overcome the basic ones)

In [ ]:
import umap

#target: 10 dimensions
reducer = umap.UMAP(
    n_neighbors=30,
    n_components=10,
    metric='cosine',
    low_memory=True,
    random_state=42
)
X_reduced = reducer.fit_transform(X)

#clean ram
del reducer
if 'X' in locals(): del X
if 'embeddings' in locals(): del embeddings
gc.collect()

X_reduced

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


array([[ 9.675132  ,  0.46914417,  6.4958997 , ...,  6.025482  ,
         6.071676  , 10.658448  ],
       [ 2.474978  ,  6.9371257 , 10.337155  , ...,  3.6248062 ,
         5.0337667 ,  4.3859797 ],
       [ 2.4561434 ,  0.2491199 ,  9.605889  , ...,  1.6305408 ,
         7.063405  ,  5.1906996 ],
       ...,
       [10.3892145 ,  0.7250388 ,  6.4655194 , ...,  6.1432137 ,
         6.2285423 , 10.521758  ],
       [ 9.781162  ,  4.195625  ,  4.7080827 , ...,  4.221271  ,
         4.6071057 ,  6.5750246 ],
       [ 8.94798   ,  1.3242997 ,  9.680245  , ...,  3.8729608 ,
         4.298156  ,  5.440892  ]], dtype=float32)

#Step 5: Clustering with HDBSCAN

In [ ]:
import hdbscan

clusterer = hdbscan.HDBSCAN(
    min_cluster_size=2000,
    min_samples=100,
    cluster_selection_epsilon=0.05,
    cluster_selection_method="leaf",
    alpha=1.2,
    prediction_data=True
)

cluster_labels = clusterer.fit_predict(X_reduced)
df_final['linguistic_cluster'] = cluster_labels

df_final

,Asset,Scrape date,Ownership history,Bids,Resale,Highest bid,grado_asset,length,n_alpha,n_digit,...,emb_1015,emb_1016,emb_1017,emb_1018,emb_1019,emb_1020,emb_1021,emb_1022,emb_1023,linguistic_cluster
0,@dubaihouse,2025-09-30 02:34:42,"[{'Owner': 'conig.t.me', 'Date of Sale': '2024...",[],None,NaN,1.0,10,10,0,...,-0.043667,-0.011217,0.023220,-0.039585,-0.025285,0.020210,0.002030,-0.039038,-0.021936,16
1,@totln,2025-11-16 20:42:28,[{'Owner': 'UQBB5t1vKAJMgLj-5cGHXhxqweHSxNkm0z...,[],None,NaN,1.0,5,5,0,...,-0.033046,0.006998,-0.020753,-0.008787,0.011834,-0.014221,-0.033157,-0.023512,-0.025341,0
2,@hangzhoubot,2025-11-13 06:50:16,"[{'Owner': 'mahua.t.me', 'Date of Sale': '2025...",[],None,NaN,4.0,11,11,0,...,-0.018247,0.011069,0.046366,-0.025089,-0.044691,-0.005340,0.034371,-0.033242,0.027659,3
3,@gssssss,2025-11-13 06:34:48,[{'Owner': 'UQAVWbDUur7WDLwT71dzKf5RDZfTUHTnWE...,[],None,NaN,2.0,7,7,0,...,-0.037786,0.024488,0.016801,-0.052073,0.012337,0.012308,0.022585,-0.009809,-0.003181,17
4,@kefu586,2025-11-13 09:12:22,[{'Owner': 'UQBC6_I2vvLCUfwNnfSGlrVTmWG7PO1j37...,[],None,NaN,1.0,7,4,3,...,-0.016200,-0.003957,0.025714,-0.046182,-0.024785,0.044084,-0.010170,-0.023104,0.078489,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110676,@toeiloilo,2025-11-16 20:07:25,[{'Owner': 'UQC3cnouK2zl0IYDZXoFLTGPOsfr9byl-G...,[],None,NaN,2.0,9,9,0,...,-0.013383,0.026089,-0.003549,-0.029183,0.036382,-0.016910,-0.025597,-0.005268,0.012683,17
110677,@lvjing,2025-11-14 11:52:34,"[{'Owner': 'tiezi.t.me', 'Date of Sale': '2023...",[],None,NaN,3.0,6,6,0,...,-0.020955,0.030257,-0.011262,-0.037203,-0.031983,0.018343,-0.004952,0.005525,0.032474,2
110678,@milklover,2025-11-14 13:10:00,"[{'Owner': 'sara88.t.me', 'Date of Sale': '202...",[],None,NaN,8.0,9,9,0,...,-0.001544,0.032280,0.015610,0.016981,0.016153,-0.013385,0.038522,-0.012018,0.055867,15
110680,@daili888,2025-11-13 03:01:33,"[{'Owner': 'tantan.t.me', 'Date of Sale': '202...",[],None,NaN,1.0,8,5,3,...,-0.051146,0.011272,0.038831,-0.018764,0.006023,0.063633,0.007415,0.018738,0.053327,18


In [ ]:
#basic clusters exploration
n_clusters = len(set(cluster_labels))-1 #cluster -1 is noise
print(f"number of clusters: {n_clusters}")
print(f"noise assets: {(cluster_labels == -1).sum():,} ({((cluster_labels == -1).sum()/len(df_final)*100):.1f}%)")

number of clusters: 20
noise assets: 25,582 (24.9%)


In [ ]:
cluster_counts = df_final['linguistic_cluster'].value_counts().sort_index()

cluster_summary = pd.DataFrame({
    'Cluster_ID': cluster_counts.index,
    'Size': cluster_counts.values,
    'Percentage': (cluster_counts.values / len(df_final) * 100).round(2)
})

#cluster -1 is noise
noise = cluster_summary[cluster_summary['Cluster_ID'] == -1]
real_clusters = cluster_summary[cluster_summary['Cluster_ID'] != -1]

print(f"total clusters: {len(real_clusters)}")
print(f"total assets: {len(df_final):,}")
print("\n")
print(f"noise: {noise['Size'].sum() if not noise.empty else 0} asset ({noise['Percentage'].sum() if not noise.empty else 0}%)")
print("\n")
print("top10 clusters (by size):")
print(real_clusters.sort_values(by='Size', ascending=False).head(10).to_string(index=False))

total clusters: 20
total assets: 102,649


noise: 25582 asset (24.92%)


top10 clusters (by size):
 Cluster_ID  Size  Percentage
          4  8435        8.22
          0  5704        5.56
         17  5158        5.02
          2  5124        4.99
          8  5063        4.93
          5  4974        4.85
          1  4842        4.72
         19  4357        4.24
          3  4093        3.99
          9  3838        3.74


#Intermediary step: save the df and the graph. Step 6 and 7 will be done on other notebooks

In [ ]:
df_final

,Asset,Scrape date,Ownership history,Bids,Resale,Highest bid,grado_asset,length,n_alpha,n_digit,...,emb_1015,emb_1016,emb_1017,emb_1018,emb_1019,emb_1020,emb_1021,emb_1022,emb_1023,linguistic_cluster
0,@dubaihouse,2025-09-30 02:34:42,"[{'Owner': 'conig.t.me', 'Date of Sale': '2024...",[],None,NaN,1.0,10,10,0,...,-0.043667,-0.011217,0.023220,-0.039585,-0.025285,0.020210,0.002030,-0.039038,-0.021936,16
1,@totln,2025-11-16 20:42:28,[{'Owner': 'UQBB5t1vKAJMgLj-5cGHXhxqweHSxNkm0z...,[],None,NaN,1.0,5,5,0,...,-0.033046,0.006998,-0.020753,-0.008787,0.011834,-0.014221,-0.033157,-0.023512,-0.025341,0
2,@hangzhoubot,2025-11-13 06:50:16,"[{'Owner': 'mahua.t.me', 'Date of Sale': '2025...",[],None,NaN,4.0,11,11,0,...,-0.018247,0.011069,0.046366,-0.025089,-0.044691,-0.005340,0.034371,-0.033242,0.027659,3
3,@gssssss,2025-11-13 06:34:48,[{'Owner': 'UQAVWbDUur7WDLwT71dzKf5RDZfTUHTnWE...,[],None,NaN,2.0,7,7,0,...,-0.037786,0.024488,0.016801,-0.052073,0.012337,0.012308,0.022585,-0.009809,-0.003181,17
4,@kefu586,2025-11-13 09:12:22,[{'Owner': 'UQBC6_I2vvLCUfwNnfSGlrVTmWG7PO1j37...,[],None,NaN,1.0,7,4,3,...,-0.016200,-0.003957,0.025714,-0.046182,-0.024785,0.044084,-0.010170,-0.023104,0.078489,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
110676,@toeiloilo,2025-11-16 20:07:25,[{'Owner': 'UQC3cnouK2zl0IYDZXoFLTGPOsfr9byl-G...,[],None,NaN,2.0,9,9,0,...,-0.013383,0.026089,-0.003549,-0.029183,0.036382,-0.016910,-0.025597,-0.005268,0.012683,17
110677,@lvjing,2025-11-14 11:52:34,"[{'Owner': 'tiezi.t.me', 'Date of Sale': '2023...",[],None,NaN,3.0,6,6,0,...,-0.020955,0.030257,-0.011262,-0.037203,-0.031983,0.018343,-0.004952,0.005525,0.032474,2
110678,@milklover,2025-11-14 13:10:00,"[{'Owner': 'sara88.t.me', 'Date of Sale': '202...",[],None,NaN,8.0,9,9,0,...,-0.001544,0.032280,0.015610,0.016981,0.016153,-0.013385,0.038522,-0.012018,0.055867,15
110680,@daili888,2025-11-13 03:01:33,"[{'Owner': 'tantan.t.me', 'Date of Sale': '202...",[],None,NaN,1.0,8,5,3,...,-0.051146,0.011272,0.038831,-0.018764,0.006023,0.063633,0.007415,0.018738,0.053327,18


In [ ]:
G_proj_final

In [ ]:
df_final.to_csv('df_step5.csv', index=False)
nx.write_weighted_edgelist(G_proj_final, 'graph_step5.edgelist')